# Section 4.5 Extended Monte Carlo Simulations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hfallahgoul/HDL-JoE-replication/blob/main/03_Sim45_Monte_Carlo_Extensions.ipynb)

This notebook runs the full-scale Section 4.5 Monte Carlo simulations under i.i.d. and AR(1) DGPs, evaluating out-of-sample $R^2$ and annualized Sharpe ratios (sign and linear timing) across training window lengths $T \in [6, 398]$ and population signal levels ($R^2 \in \{1.0\%, 2.3\%, 5.0\%\}$).

It produces Figures A–D (`fig_r2_baseline.pdf`, `fig_sharpe.pdf`, `fig_iid_vs_ar1.pdf`, `fig_sensitivity.pdf`) and the exact summary statistics for the paper.

# Importing WG data-set

In [1]:
import os
import sys

CSV_NAME = 'PredictorData2021.csv'
GITHUB_RAW_URL = f'https://raw.githubusercontent.com/hfallahgoul/HDL-JoE-replication/main/{CSV_NAME}'
DRIVE_FALLBACK = f'/content/drive/MyDrive/HDL_Finance/{CSV_NAME}'

def get_data_path(csv_name=CSV_NAME):
    if os.path.exists(csv_name):
        return csv_name
    elif os.path.exists(DRIVE_FALLBACK):
        return DRIVE_FALLBACK
    else:
        try:
            import urllib.request
            print(f'Downloading {csv_name} from GitHub repository...')
            urllib.request.urlretrieve(GITHUB_RAW_URL, csv_name)
            if os.path.exists(csv_name):
                print(f'Download complete: {csv_name}')
                return csv_name
        except Exception as e:
            print(f'Note: Automatic download from GitHub failed ({e}).')
    return csv_name

csv_path = get_data_path()
print(f'Using dataset path: {csv_path}')


The file exists at: /content/drive/MyDrive/HDL_Finance/PredictorData2021.csv


In [1]:
import argparse
import os
import sys

import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV


# ----------------------------------------------------------------- data
GW_COLS = ['D12', 'E12', 'Index', 'BAA', 'AAA', 'lty', 'tbl', 'corpr',
           'ltr', 'CRSP_SPvw', 'Rfree']
PREDICTORS = ['dfy', 'infl', 'svar', 'de', 'lty', 'tms', 'tbl', 'dfr',
              'dp', 'dy', 'ltr', 'ep', 'b/m', 'ntis', 'lag_exret']


def load_predictors(csv_path):
    df = pd.read_csv(csv_path)
    df['Date'] = pd.to_datetime(df['yyyymm'], format='%Y%m')
    df = df.set_index('Date')
    for col in GW_COLS:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['dp'] = np.log(df['D12']) - np.log(df['Index'])
    df['dy'] = np.log(df['D12']) - np.log(df['Index'].shift(1))
    df['ep'] = np.log(df['E12']) - np.log(df['Index'])
    df['de'] = np.log(df['D12']) - np.log(df['E12'])
    df['dfy'] = df['BAA'] - df['AAA']
    df['tms'] = df['lty'] - df['tbl']
    df['dfr'] = df['corpr'] - df['ltr']
    df['exret'] = df['CRSP_SPvw'] - df['Rfree']
    df['lag_exret'] = df['exret'].shift(1)
    return df[PREDICTORS].ffill().dropna().astype(np.float64)


# ------------------------------------------------------------- features
def make_rff(X, omega, b, gamma):
    arg = gamma * (X @ omega) + b[np.newaxis, :]
    return np.sqrt(2.0) * np.hstack([np.cos(arg), np.sin(arg)])


# ---------------------------------------------------------- input draws
def draw_iid(rng, n, mu, cov):
    return rng.multivariate_normal(mu, cov, size=n)


class AR1Sampler:
    """Stationary diagonal-Phi VAR(1) whose unconditional covariance matches cov.

    x_t - mu = Phi (x_{t-1} - mu) + u_t,  u_t ~ N(0, Sigma_u),
    Sigma_u = cov - Phi cov Phi' projected to the PSD cone (eigenvalue clip).
    """

    def __init__(self, mu, cov, phis, burn=200):
        self.mu, self.cov, self.burn = mu, cov, burn
        self.phi = np.clip(phis, 0.0, 0.99)
        D = np.diag(self.phi)
        Su = cov - D @ cov @ D
        w, V = np.linalg.eigh((Su + Su.T) / 2.0)
        w = np.clip(w, 1e-12, None)
        self.chol = V @ np.diag(np.sqrt(w))          # Sigma_u^{1/2}

    def draw(self, rng, n):
        k = len(self.mu)
        x = np.zeros(k)                              # start at the mean
        out = np.empty((n, k))
        total = self.burn + n
        eps = rng.standard_normal((total, k)) @ self.chol.T
        for t in range(total):
            x = self.phi * x + eps[t]
            if t >= self.burn:
                out[t - self.burn] = x
        return out + self.mu


# ------------------------------------------------------------- metrics
def sharpe_annualized(strategy_returns):
    s = np.std(strategy_returns)
    if s == 0:
        return np.nan
    return np.mean(strategy_returns) / s * np.sqrt(12.0)


def summarize(values):
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    med = np.median(v)
    boot = np.array([np.median(np.random.default_rng(i).choice(v, size=v.size))
                     for i in range(500)])
    return dict(median=med, p25=np.percentile(v, 25), p75=np.percentile(v, 75),
                mean=np.mean(v), se_mean=np.std(v) / np.sqrt(v.size),
                ci_lo=np.percentile(boot, 2.5), ci_hi=np.percentile(boot, 97.5),
                n=v.size)


# ----------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--variant', choices=['iid', 'ar1'], default='iid')
    ap.add_argument('--target-r2', type=float, default=0.023)
    ap.add_argument('--P', type=int, default=2000)
    ap.add_argument('--trials', type=int, default=50)
    ap.add_argument('--n-test', type=int, default=1000)
    ap.add_argument('--gamma', type=float, default=2.0)
    ap.add_argument('--t-grid', default='proto',
                    help="'proto' = 8 points, 'full' = 6:398:8, "
                         "or comma-separated ints")
    ap.add_argument('--seed', type=int, default=42)
    ap.add_argument('--csv', default=None)
    ap.add_argument('--outdir', default=None)
    args = ap.parse_args([])

    here = os.getcwd()
    # Use the absolute path provided by the user
    csv = get_data_path()
    outdir = args.outdir or os.path.join(here, 'results')
    os.makedirs(outdir, exist_ok=True)

    if args.t_grid == 'proto':
        t_grid = [6, 12, 25, 50, 100, 200, 300, 400]
    elif args.t_grid == 'full':
        t_grid = list(range(6, 400, 8))
    else:
        t_grid = [int(s) for s in args.t_grid.split(',')]

    X = load_predictors(csv).values
    mu_x, cov_x = X.mean(axis=0), np.cov(X, rowvar=False)
    K = X.shape[1]

    # lag-1 autocorrelations for the AR(1) variant
    phis = np.array([pd.Series(X[:, j]).autocorr(1) for j in range(K)])

    rng = np.random.default_rng(args.seed)
    omega = rng.standard_normal((K, args.P // 2))
    b = rng.uniform(0, 2 * np.pi, args.P // 2)
    beta = np.zeros(args.P)
    active = rng.choice(args.P, size=50, replace=False)
    beta[active] = rng.standard_normal(50)
    beta /= np.linalg.norm(beta)

    # calibration of sigma_eps (on iid draws)
    Z_cal = make_rff(draw_iid(rng, 10_000, mu_x, cov_x), omega, b, args.gamma)
    signal_var = np.var(Z_cal @ beta)
    sigma_eps = np.sqrt(signal_var * (1.0 / args.target_r2 - 1.0))

    sampler = (AR1Sampler(mu_x, cov_x, phis) if args.variant == 'ar1' else None)
    alphas = np.logspace(-3, 4, 20)

    rows = []
    for T in t_grid:
        r2s, sr_sign, sr_lin = [], [], []
        for _ in range(args.trials):
            if sampler is None:
                X_tr = draw_iid(rng, T, mu_x, cov_x)
                X_te = draw_iid(rng, args.n_test, mu_x, cov_x)
            else:
                X_tr = sampler.draw(rng, T)
                X_te = sampler.draw(rng, args.n_test)
            Z_tr = make_rff(X_tr, omega, b, args.gamma)
            Z_te = make_rff(X_te, omega, b, args.gamma)
            y_tr = Z_tr @ beta + rng.normal(0, sigma_eps, T)
            y_te = Z_te @ beta + rng.normal(0, sigma_eps, args.n_test)

            model = RidgeCV(alphas=alphas)
            model.fit(Z_tr, y_tr)
            y_hat = model.predict(Z_te)

            r2s.append(1.0 - np.sum((y_te - y_hat) ** 2)
                       / np.sum((y_te - y_te.mean()) ** 2))
            sr_sign.append(sharpe_annualized(np.sign(y_hat) * y_te))
            w = y_hat / (np.std(y_hat) + 1e-12)      # unit-vol linear timing
            sr_lin.append(sharpe_annualized(w * y_te))

        for name, vals in [('oos_r2', r2s), ('sr_sign', sr_sign),
                           ('sr_linear', sr_lin)]:
            row = dict(variant=args.variant, target_r2=args.target_r2,
                       P=args.P, T=T, metric=name, **summarize(vals))
            rows.append(row)
        print(f"T={T:3d}  medR2={np.median(r2s):+.4f}  "
              f"medSR(sign)={np.median(sr_sign):+.3f}  "
              f"medSR(lin)={np.median(sr_lin):+.3f}", flush=True)

    tag = f"{args.variant}_r2-{args.target_r2:.3f}_P{args.P}"
    out = os.path.join(outdir, f"sim45_{tag}.csv")
    pd.DataFrame(rows).to_csv(out, index=False)
    print("saved:", out)

if __name__ == '__main__':
    main()


T=  6  medR2=-0.1534  medSR(sign)=+0.057  medSR(lin)=+0.101
T= 12  medR2=-0.1127  medSR(sign)=+0.082  medSR(lin)=+0.080
T= 25  medR2=-0.0416  medSR(sign)=+0.016  medSR(lin)=+0.083
T= 50  medR2=-0.0366  medSR(sign)=+0.141  medSR(lin)=+0.129
T=100  medR2=-0.0308  medSR(sign)=+0.106  medSR(lin)=+0.119
T=200  medR2=-0.0202  medSR(sign)=+0.158  medSR(lin)=+0.212
T=300  medR2=-0.0144  medSR(sign)=+0.200  medSR(lin)=+0.224
T=400  medR2=-0.0133  medSR(sign)=+0.169  medSR(lin)=+0.222
saved: /content/results/sim45_iid_r2-0.023_P2000.csv


**Sim45 Full**

# All Models DGP: IID, AR,...

In [1]:
# ============================================================================
# FULL-SCALE Section 4.5 Monte Carlo — Google Colab version
# Monte Carlo Simulation Study: R^2, Sharpe ratio, iid/AR(1), and sensitivity
#
# HOW TO USE (Colab):
#   1. Upload PredictorData2021.csv (or the 2022 vintage) to
#      MyDrive/HDL_Finance/  — same location your other notebooks use.
#   2. Paste this entire file into ONE Colab cell and run, or upload it and
#      run  %run colab_sim45_full.py
#   3. Results, figures, and checkpoints are written to
#      MyDrive/HDL_Finance/sim45_full/   — safe to disconnect and re-run:
#      completed trials are detected and skipped (checkpoint/resume).
#
# DESIGN: DGP, features, signal,
# calibration, lambda grid, and R^2 benchmark. Fast nested implementation:
#   NESTED_TRIALS=True draws, per trial, ONE training path of length
#   max(T_GRID) and ONE test set, then evaluates every T on the first T rows.
#   Feature generation drops by ~50x (hours instead of days). The marginal
#   distribution of each per-T statistic is IDENTICAL to independent
#   redraws; only cross-T dependence changes, which nothing we report uses.
#   Set NESTED_TRIALS=False for fully independent redraws.
#
# Runtime guide (Colab CPU, float32): ~5-9 s/trial in nested mode
#   => ~20-30 min per variant, ~2 h for all four. High-RAM runtime helps.
# ============================================================================

import os, sys, time
import numpy as np
import pandas as pd

# ----------------------------- configuration -------------------------------
IN_COLAB   = 'google.colab' in sys.modules
DRIVE_DIR  = '/content/drive/MyDrive/HDL_Finance'
CSV_NAME   = 'PredictorData2021.csv'        # <- switch to 2022 vintage if used
OUT_SUB    = 'sim45_full'

P_SIM      = 12_000
N_TRIALS   = 200
N_TEST     = 2_000
GAMMA      = 2.0
T_GRID     = list(range(6, 400, 8))         # 6, 14, ..., 398
ALPHAS     = np.logspace(-3, 4, 20)         # 7 orders of magnitude
SEED       = 42
NESTED_TRIALS = True
DTYPE      = np.float32
CHECKPOINT_EVERY = 10                        # trials between checkpoint saves

VARIANTS   = [('iid', 0.023), ('ar1', 0.023), ('iid', 0.010), ('iid', 0.050)]

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
CSV_PATH = get_data_path(CSV_NAME)
OUTDIR   = os.path.join(DRIVE_DIR, OUT_SUB) if (IN_COLAB and os.path.exists(DRIVE_DIR)) else OUT_SUB
os.makedirs(OUTDIR, exist_ok=True)

from sklearn.linear_model import RidgeCV

# ------------------------------- data --------------------------------------
GW_COLS = ['D12','E12','Index','BAA','AAA','lty','tbl','corpr','ltr','CRSP_SPvw','Rfree']
PREDS   = ['dfy','infl','svar','de','lty','tms','tbl','dfr','dp','dy','ltr','ep','b/m','ntis','lag_exret']

def load_predictors(path):
    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(df['yyyymm'], format='%Y%m'); df = df.set_index('Date')
    for c in GW_COLS: df[c] = pd.to_numeric(df[c], errors='coerce')
    df['dp']  = np.log(df['D12']) - np.log(df['Index'])
    df['dy']  = np.log(df['D12']) - np.log(df['Index'].shift(1))
    df['ep']  = np.log(df['E12']) - np.log(df['Index'])
    df['de']  = np.log(df['D12']) - np.log(df['E12'])
    df['dfy'] = df['BAA'] - df['AAA']; df['tms'] = df['lty'] - df['tbl']
    df['dfr'] = df['corpr'] - df['ltr']
    df['exret'] = df['CRSP_SPvw'] - df['Rfree']; df['lag_exret'] = df['exret'].shift(1)
    return df[PREDS].ffill().dropna().astype(np.float64)

def make_rff(X, omega, b, gamma):
    arg = gamma * (X @ omega) + b[np.newaxis, :]
    return (np.sqrt(2.0) * np.hstack([np.cos(arg), np.sin(arg)])).astype(DTYPE)

class AR1Sampler:
    """Diagonal-Phi VAR(1), unconditional covariance matched to cov (PSD-projected)."""
    def __init__(self, mu, cov, phis, burn=200):
        self.mu, self.burn = mu, burn
        self.phi = np.clip(phis, 0.0, 0.99)
        D  = np.diag(self.phi); Su = cov - D @ cov @ D
        w, V = np.linalg.eigh((Su + Su.T) / 2.0)
        self.chol = V @ np.diag(np.sqrt(np.clip(w, 1e-12, None)))
    def draw(self, rng, n):
        k = len(self.mu); x = np.zeros(k); out = np.empty((n, k))
        eps = rng.standard_normal((self.burn + n, k)) @ self.chol.T
        for t in range(self.burn + n):
            x = self.phi * x + eps[t]
            if t >= self.burn: out[t - self.burn] = x
        return out + self.mu

def sharpe_ann(s):
    sd = np.std(s); return np.nan if sd == 0 else np.mean(s) / sd * np.sqrt(12.0)

# ------------------------------ one variant --------------------------------
def run_variant(variant, target_r2, Xdata):
    tag  = f"{variant}_r2-{target_r2:.3f}_P{P_SIM}"
    rawf = os.path.join(OUTDIR, f"raw_{tag}.csv")       # per-trial rows (checkpoint)
    done = 0
    if os.path.exists(rawf):
        done = pd.read_csv(rawf)['trial'].nunique()
        print(f"[{tag}] resuming: {done}/{N_TRIALS} trials already on disk")
        if done >= N_TRIALS: return rawf
    mu_x, cov_x = Xdata.mean(axis=0), np.cov(Xdata, rowvar=False)
    K = Xdata.shape[1]
    phis = np.array([pd.Series(Xdata[:, j]).autocorr(1) for j in range(K)])

    # --- fixed objects: reproducible RNG initialization ---
    rng   = np.random.default_rng(SEED)
    omega = rng.standard_normal((K, P_SIM // 2))
    b     = rng.uniform(0, 2 * np.pi, P_SIM // 2)
    beta  = np.zeros(P_SIM)
    act   = rng.choice(P_SIM, size=50, replace=False)
    beta[act] = rng.standard_normal(50)
    beta /= np.linalg.norm(beta); beta = beta.astype(DTYPE)

    Z_cal = make_rff(rng.multivariate_normal(mu_x, cov_x, size=10_000), omega, b, GAMMA)
    sigma_eps = float(np.sqrt(np.var(Z_cal @ beta) * (1/target_r2 - 1)))
    del Z_cal
    sampler = AR1Sampler(mu_x, cov_x, phis) if variant == 'ar1' else None
    Tmax = max(T_GRID)

    # advance rng deterministically past completed trials on resume
    trial_rngs = [np.random.default_rng([SEED, 1000 + i]) for i in range(N_TRIALS)]

    t0 = time.time(); buf = []
    for tr in range(done, N_TRIALS):
        r = trial_rngs[tr]
        if NESTED_TRIALS:
            X_pool = sampler.draw(r, Tmax) if sampler else r.multivariate_normal(mu_x, cov_x, size=Tmax)
            X_te   = sampler.draw(r, N_TEST) if sampler else r.multivariate_normal(mu_x, cov_x, size=N_TEST)
            Z_pool = make_rff(X_pool, omega, b, GAMMA)
            Z_te   = make_rff(X_te,   omega, b, GAMMA)
            y_pool = Z_pool @ beta + r.normal(0, sigma_eps, Tmax).astype(DTYPE)
            y_te   = Z_te   @ beta + r.normal(0, sigma_eps, N_TEST).astype(DTYPE)
        for T in T_GRID:
            if NESTED_TRIALS:
                Z_tr, y_tr = Z_pool[:T], y_pool[:T]
            else:
                X_tr = sampler.draw(r, T) if sampler else r.multivariate_normal(mu_x, cov_x, size=T)
                X_t2 = sampler.draw(r, N_TEST) if sampler else r.multivariate_normal(mu_x, cov_x, size=N_TEST)
                Z_tr = make_rff(X_tr, omega, b, GAMMA); Z_te = make_rff(X_t2, omega, b, GAMMA)
                y_tr = Z_tr @ beta + r.normal(0, sigma_eps, T).astype(DTYPE)
                y_te = Z_te @ beta + r.normal(0, sigma_eps, N_TEST).astype(DTYPE)
            m = RidgeCV(alphas=ALPHAS); m.fit(Z_tr, y_tr)
            yh = m.predict(Z_te)
            r2 = 1.0 - np.sum((y_te - yh)**2) / np.sum((y_te - y_te.mean())**2)
            buf.append(dict(trial=tr, T=T,
                            oos_r2=r2,
                            sr_sign=sharpe_ann(np.sign(yh) * y_te),
                            sr_linear=sharpe_ann(yh / (np.std(yh) + 1e-12) * y_te),
                            alpha_sel=float(m.alpha_)))
        if (tr + 1) % CHECKPOINT_EVERY == 0 or tr == N_TRIALS - 1:
            pd.DataFrame(buf).to_csv(rawf, mode='a', header=not os.path.exists(rawf), index=False)
            buf = []
            el = time.time() - t0
            print(f"[{tag}] trial {tr+1}/{N_TRIALS}  ({el/60:.1f} min elapsed, "
                  f"~{el/(tr+1-done)*(N_TRIALS-tr-1)/60:.0f} min left)", flush=True)
    return rawf

# ------------------------------ aggregation --------------------------------
def summarize(rawf, variant, target_r2):
    raw = pd.read_csv(rawf); rows = []
    for T, g in raw.groupby('T'):
        for met in ['oos_r2', 'sr_sign', 'sr_linear']:
            v = g[met].dropna().values
            boot = np.array([np.median(np.random.default_rng(i).choice(v, v.size)) for i in range(500)])
            rows.append(dict(variant=variant, target_r2=target_r2, T=T, metric=met,
                             median=np.median(v), p25=np.percentile(v, 25), p75=np.percentile(v, 75),
                             ci_lo=np.percentile(boot, 2.5), ci_hi=np.percentile(boot, 97.5),
                             mean=v.mean(), n=v.size,
                             med_alpha=g['alpha_sel'].median()))
    return pd.DataFrame(rows)

# --------------------------------- run -------------------------------------
Xdata = load_predictors(CSV_PATH).values
print(f"data: {Xdata.shape[0]} months x {Xdata.shape[1]} predictors")
all_sum = []
for variant, tr2 in VARIANTS:
    rawf = run_variant(variant, tr2, Xdata)
    all_sum.append(summarize(rawf, variant, tr2))
S = pd.concat(all_sum, ignore_index=True)
S.to_csv(os.path.join(OUTDIR, 'sim45_full_summary.csv'), index=False)

# -------------------------------- figures ----------------------------------
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
TEAL, ORANGE, BLUE, BROWN = '#00695c', '#bf360c', '#1565c0', '#6d4c41'

def band(ax, d, color, label):
    ax.plot(d['T'], d['median'], '-', color=color, lw=2, label=label)
    ax.fill_between(d['T'], d['ci_lo'], d['ci_hi'], color=color, alpha=0.18)

# Fig A: baseline R2 with bands (updates Figure 6)
fig, ax = plt.subplots(figsize=(9, 5.2))
band(ax, S[(S.variant=='iid') & (S.target_r2==0.023) & (S.metric=='oos_r2')], TEAL, 'median OOS $R^2$ (95% CI of median)')
ax.axhline(0, color='k', lw=1, ls='--'); ax.axhline(0.023, color='gray', lw=1, ls=':', label='population $R^2$ = 2.3%')
ax.axvline(344, color='gray', lw=1, ls='-.', label='$T_{crit} \\approx 344$')
ax.set_xlabel('training window $T$'); ax.set_ylabel('OOS $R^2$'); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(os.path.join(OUTDIR, 'fig_r2_baseline.pdf')); plt.close(fig)

# Fig B: Sharpe panel (new)
fig, ax = plt.subplots(figsize=(9, 5.2))
band(ax, S[(S.variant=='iid') & (S.target_r2==0.023) & (S.metric=='sr_sign')],   TEAL,  'sign timing')
band(ax, S[(S.variant=='iid') & (S.target_r2==0.023) & (S.metric=='sr_linear')], BLUE,  'linear timing')
ax.axhline(0, color='k', lw=1, ls='--')
ax.set_xlabel('training window $T$'); ax.set_ylabel('annualized Sharpe ratio'); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(os.path.join(OUTDIR, 'fig_sharpe.pdf')); plt.close(fig)

# Fig C: iid vs AR(1)
fig, ax = plt.subplots(figsize=(9, 5.2))
band(ax, S[(S.variant=='iid') & (S.target_r2==0.023) & (S.metric=='oos_r2')], TEAL,   'i.i.d. inputs')
band(ax, S[(S.variant=='ar1') & (S.target_r2==0.023) & (S.metric=='oos_r2')], ORANGE, 'AR(1) inputs (fitted $\\phi_j$)')
ax.axhline(0, color='k', lw=1, ls='--')
ax.set_xlabel('training window $T$'); ax.set_ylabel('OOS $R^2$'); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(os.path.join(OUTDIR, 'fig_iid_vs_ar1.pdf')); plt.close(fig)

# Fig D: calibration sensitivity
fig, ax = plt.subplots(figsize=(9, 5.2))
for tr2, c in [(0.010, BROWN), (0.023, TEAL), (0.050, BLUE)]:
    band(ax, S[(S.variant=='iid') & (S.target_r2==tr2) & (S.metric=='oos_r2')], c, f'population $R^2$ = {tr2*100:.1f}%')
ax.axhline(0, color='k', lw=1, ls='--')
ax.set_xlabel('training window $T$'); ax.set_ylabel('OOS $R^2$'); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(os.path.join(OUTDIR, 'fig_sensitivity.pdf')); plt.close(fig)

# --------------------- numbers for the [FILL] markers ----------------------
def med(v, t, m, tr2=0.023):
    d = S[(S.variant==v) & (S.target_r2==tr2) & (S.metric==m) & (S['T']==t)]
    return float(d['median'].iloc[0]) if len(d) else np.nan

print("\n================ SUMMARY SIMULATION METRICS ================")
print(f"SR(linear) T=14: {med('iid',14,'sr_linear'):+.2f}"
      f" -> T=398: {med('iid',398,'sr_linear'):+.2f}; "
      f"SR(sign) T=14: {med('iid',14,'sr_sign'):+.2f} -> T=398: {med('iid',398,'sr_sign'):+.2f}; "
      f"R2 stays negative: max median over T = "
      f"{S[(S.variant=='iid')&(S.target_r2==0.023)&(S.metric=='oos_r2')]['median'].max():+.4f}")
print(f"AR(1) vs iid, median R2 at T=6: "
      f"{med('ar1',6,'oos_r2'):+.3f} vs {med('iid',6,'oos_r2'):+.3f}; at T=14: "
      f"{med('ar1',14,'oos_r2'):+.3f} vs {med('iid',14,'oos_r2'):+.3f}; at T=398: "
      f"{med('ar1',398,'oos_r2'):+.3f} vs {med('iid',398,'oos_r2'):+.3f}")
neg5 = S[(S.variant=='iid') & (S.target_r2==0.050) & (S.metric=='oos_r2') & (S['median'] < 0)]
print(f"at 5% signal, median R2 negative through T="
      f"{int(neg5['T'].max()) if len(neg5) else 'none'}; at 1%: unchanged pattern "
      f"(median at T=398: {med('iid',398,'oos_r2',0.010):+.3f})")
print("=================================================================")
print("Outputs in:", OUTDIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
data: 1141 months x 15 predictors
[iid_r2-0.023_P12000] resuming: 200/200 trials already on disk
[ar1_r2-0.023_P12000] resuming: 200/200 trials already on disk
[iid_r2-0.010_P12000] resuming: 200/200 trials already on disk
[iid_r2-0.050_P12000] resuming: 200/200 trials already on disk

================ SUMMARY SIMULATION METRICS ================
SR(linear) T=14: +0.13 -> T=398: +0.25; SR(sign) T=14: +0.10 -> T=398: +0.23; R2 stays negative: max median over T = -0.0612
AR(1) vs iid, median R2 at T=6: -0.249 vs -0.186; at T=14: -0.120 vs -0.155; at T=398: -0.051 vs -0.061
at 5% signal, median R2 negative through T=398; at 1%: unchanged pattern (median at T=398: -0.074)
Outputs in: /content/drive/MyDrive/HDL_Finance/sim45_full
